# Multi-Task Hard-Sharing Multi-Branch CNN

Targets:

- `gesture_action` primary head
- `orientation` auxiliary head
- `phase` auxiliary head

Model:

- shared multi-branch CNN backbone
- attention / BiGRU / none fusion
- three task heads
- uncertainty-weighted task losses
- multi-domain sensor extraction
- handedness and upside-down correction inside the pipeline
- optional Kalman / EKF-style smoothing and dead-reckoning features


In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.metrics import f1_score, classification_report

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
except Exception:
    print("scikit-optimize is not installed. Bayesian optimization will not be available.")

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/hardsharer/')
sys.path.append('/kaggle/input/cmi-competition-code')

import data_utils
import utils_multitask_hardsharing as utils


In [ ]:
# ============================================================
# Config
# ============================================================

search_mode = "grid"      # "grid" or "bayesian"
random_state = 42
n_splits = 2
n_iter = 10
n_jobs = 1

use_subject_holdout = False
holdout_size = 0.3
train_size = 0.7     # used only when use_subject_holdout=False
chosen_orientation = None

pipe_name = "sequence_builder"
corrector_name = "sensor_corrector"
classifier_name = "classifier"

primary_target = "gesture_action"
orientation_target = "orientation"

# Reuse the existing third "phase" head as a position/location head.
# No utils rewrite needed.
position_target = "gesture_position"
phase_target = position_target

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")


In [3]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print(raw_train_df.shape)
print(raw_test_df.shape)
print(train_demo_df.shape)
print(test_demo_df.shape)


Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
(574945, 341)
(107, 336)
(81, 8)
(2, 8)


In [4]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)
train_df.loc[:, "phase_target"] = train_df["phase"]
train_df.loc[:, "orientation_action"] = train_df["orientation"].astype(str) + "||" + train_df["gesture_action"].astype(str)

print("all sequences:", train_df["sequence_id"].nunique())
print("target sequences:", train_df.loc[train_df["sequence_type"].eq("Target"), "sequence_id"].nunique())
print("gesture_action classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].dropna().unique()))
print("orientation classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].dropna().unique()))
print("gesture_position classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_position"].dropna().unique()))
print("full gesture classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture"].dropna().unique()))


all sequences: 8151
target sequences: 5113
gesture_action classes: ['pinch skin', 'pull hair', 'pull hairline', 'scratch']
orientation classes: ['Lie on Back', 'Lie on Side - Non Dominant', 'Seated Lean Non Dom - FACE DOWN', 'Seated Straight']
gesture_position classes: ['Above ear', 'Cheek', 'Eyebrow', 'Eyelash', 'Forehead', 'Neck']
full gesture classes: ['Above ear - pull hair', 'Cheek - pinch skin', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Forehead - pull hairline', 'Forehead - scratch', 'Neck - pinch skin', 'Neck - scratch']


In [5]:
# ============================================================
# Split: subject holdout OR old train_size logic
# ============================================================

if use_subject_holdout:
    seq_meta = (
        train_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "gesture_action", "orientation", "phase"]]
        .reset_index(drop=True)
    )

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=holdout_size,
        random_state=random_state,
    )

    train_seq_idx, holdout_seq_idx = next(
        splitter.split(
            seq_meta,
            y=seq_meta["gesture"],
            groups=seq_meta["subject"],
        )
    )

    train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
    holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]

    train_sample_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
    test_sample_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

else:
    if train_size is None:
        rows = (
            (train_demo_df["adult_child"] == 1)
            & (train_demo_df["sex"] == 1)
            & (train_demo_df["handedness"] == 1)
        )

        ideal_subject_ids = (
            train_demo_df.loc[rows]
            .sort_values("elbow_to_wrist_cm", ascending=False)["subject"]
            .to_list()
        )

        train_sample_df = train_df.loc[
            train_df["subject"].isin(ideal_subject_ids)
            & train_df["sequence_type"].eq("Target")
        ].copy()

        test_sample_df = None

    elif train_size == 0:
        some_sequences = train_df["sequence_id"].unique()[10:20]

        train_sample_df = train_df.loc[
            train_df["sequence_id"].isin(some_sequences)
        ].copy()

        test_sample_df = None

    else:
        target_df = train_df.loc[train_df["sequence_type"].eq("Target")].copy()

        train_sample_df, test_sample_df = data_utils.sample_balanced_split(
            target_df,
            train_pct=train_size,
            test_pct=0.2,
            random_state=random_state,
        )

        train_sample_df = train_sample_df.copy()
        test_sample_df = test_sample_df.copy()

if chosen_orientation is not None:
    train_sample_df = train_sample_df.loc[train_sample_df["orientation"].isin(chosen_orientation)].copy()
    if test_sample_df is not None:
        test_sample_df = test_sample_df.loc[test_sample_df["orientation"].isin(chosen_orientation)].copy()

target_only_train_df = train_sample_df.loc[train_sample_df["sequence_type"].eq("Target")].copy()

target_only_holdout_df = None
if test_sample_df is not None:
    target_only_holdout_df = test_sample_df.loc[test_sample_df["sequence_type"].eq("Target")].copy()

print("use_subject_holdout:", use_subject_holdout)
print("train sequences:", train_sample_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())

if test_sample_df is not None:
    print("holdout/test sequences:", test_sample_df["sequence_id"].nunique())
    print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
    print("train subjects:", train_sample_df["subject"].nunique())
    print("holdout subjects:", test_sample_df["subject"].nunique())
    print("subject overlap:", len(set(train_sample_df["subject"]) & set(test_sample_df["subject"])))


Train: 1944 seqs | 38.0%
Test:  648 seqs  | 12.7%
use_subject_holdout: False
train sequences: 1944
target-only train sequences: 1944
holdout/test sequences: 648
target-only holdout sequences: 648
train subjects: 81
holdout subjects: 81
subject overlap: 81


In [6]:
# ============================================================
# Mapping check: orientation + gesture_action -> original gesture
# ============================================================

target_map_df = (
    target_only_train_df
    .drop_duplicates(["orientation", "gesture_action", "gesture"])
    [["orientation", "gesture_action", "gesture"]]
    .copy()
)

mapping_check = (
    target_map_df
    .groupby(["orientation", "gesture_action"])["gesture"]
    .nunique()
    .reset_index(name="n_gesture")
    .sort_values("n_gesture", ascending=False)
)

print(mapping_check.head(20))
print("max mappings per orientation/action:", mapping_check["n_gesture"].max())


                        orientation gesture_action  n_gesture
1                       Lie on Back      pull hair          3
5        Lie on Side - Non Dominant      pull hair          3
13                  Seated Straight      pull hair          3
9   Seated Lean Non Dom - FACE DOWN      pull hair          3
7        Lie on Side - Non Dominant        scratch          2
4        Lie on Side - Non Dominant     pinch skin          2
3                       Lie on Back        scratch          2
0                       Lie on Back     pinch skin          2
15                  Seated Straight        scratch          2
12                  Seated Straight     pinch skin          2
11  Seated Lean Non Dom - FACE DOWN        scratch          2
8   Seated Lean Non Dom - FACE DOWN     pinch skin          2
6        Lie on Side - Non Dominant  pull hairline          1
2                       Lie on Back  pull hairline          1
10  Seated Lean Non Dom - FACE DOWN  pull hairline          1
14      

In [7]:
# ============================================================
# Pipeline
# ============================================================

pipeline = Pipeline([
    (
        corrector_name,
        utils.SensorOrientationCorrector(
            demo_df=train_demo_df,
            apply_handedness=True,
            apply_upside_down=True,
        ),
    ),
    (
        pipe_name,
        utils.AdvancedMultiDomainSequenceExtractor(
            acc_modes=("raw", "velocity", "displacement", "jerk"),
            rotation_modes=("quaternion", "rot6d", "angular_velocity"),
            sampling_rate=10,
            compute_dt=True,
            interp_mode="linear",
            standardize="mean_std",
            tof_mode="pooled_diff",
            tof_fill_mode="nan_interpolate",
            thm_mode="centered_diff",
            use_acc_magnitude=True,
            use_linear_acc_magnitude=True,
            linear_acc_mode="baseline",
            motion_filter_mode=None,
            use_dead_reckoning=False,
        ),
    ),
    (
        classifier_name,
        utils.KerasMultiTaskMultiBranchClassifier(
            gesture_target=primary_target,
            orientation_target=orientation_target,
            phase_target=phase_target,
            maxlen=160,
            branch_filters={"acc": "64-128-256-256", "rot": "64-64", "tof": "64", "thm": "16"},
            branch_kernel_sizes={"acc": "5-5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            branch_pool_sizes={"acc": "2-2-none-none", "rot": "none-none", "tof": "none", "thm": "none"},
            fusion_mode="attention",
            attention_heads=4,
            gru_units=170,
            dense_units="64",
            dropout=0.45,
            spatial_dropout=0.1,
            learning_rate=2e-4,
            batch_size=16,
            epochs=140,
            patience=15,
            uncertainty_weighting=False,
            fixed_gesture_weight=1.0,
            fixed_orientation_weight=0.3,
            fixed_phase_weight=1.0,
            verbose=0,
            random_state=random_state,
        ),
    ),
])


In [ ]:
# ============================================================
# Search space
# ============================================================
# ============================================================
# Search space
# ============================================================

if search_mode == "bayesian":
    param_space = {
        # ============================================================
        # Sequence builder / feature extraction
        # ============================================================
        f"{pipe_name}__acc_modes": Categorical(["smoothed|velocity|displacement|jerk"]),
        f"{pipe_name}__rotation_modes": Categorical(["quaternion|euler|rot6d|angular_velocity"]),
        f"{pipe_name}__sampling_rate": Integer(5, 200),
        f"{pipe_name}__interp_mode": Categorical(["ffill"]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__tof_mode": Categorical([ "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["far_255"]),
        f"{pipe_name}__thm_mode": Categorical(["centered_diff"]),
        f"{pipe_name}__motion_filter_mode": Categorical(["extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-4, 2.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__window_size": Integer(2, 200),
        f"{pipe_name}__smooth_alpha": Categorical([0.8]),

        # ============================================================
        # Multi-task multi-branch classifier
        # ============================================================
        f"{classifier_name}__maxlen": Integer(10, 300),

        f"{classifier_name}__branch_filters": [
            #{"acc": "256-512-512-512", "rot": "256-512-512-512", "tof": "128", "thm": "32"},
            #{"acc": "256-512-512-512", "rot": "256-512-512-512", "tof": "128-256", "thm": "32"},
            #{"acc": "256-512-512-512", "rot": "256-512-512-512", "tof": "256-256", "thm": "64"},
            #{"acc": "256-512-768-768", "rot": "256-512-512-512", "tof": "128-256", "thm": "32"},
            {"acc": "256-512-512-512", "rot": "256-512-768-768", "tof": "256-256", "thm": "32"},
            # {"acc": "256-512-768-768", "rot": "256-512-768-768", "tof": "256-256-256", "thm": "64"},
            # {"acc": "256-512-1024-1024", "rot": "256-512-768-768", "tof": "256-512-768-768", "thm": "64"},
        ],

        f"{classifier_name}__branch_kernel_sizes": [
            #{"acc": "5-5-5", "rot": "5-5", "tof": "3-3", "thm": "3"},
            {"acc": "5-5-5-5", "rot": "5-5-5-5", "tof": "3-3", "thm": "3"},
           # {"acc": "7-5-5-3", "rot": "5-5-3-3", "tof": "3-3", "thm": "3"},
        ],

        f"{classifier_name}__branch_pool_sizes": [
            {"acc": "none", "rot": "none", "tof": "none", "thm": "none"},
            # {"acc": "2", "rot": "2", "tof": "none", "thm": "none"},
            # {"acc": "2", "rot": "2", "tof": "2", "thm": "none"},
        ],

        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__fusion_mode": Categorical(["attention"]),
        f"{classifier_name}__attention_heads": Integer(10, 100),
        f"{classifier_name}__gru_units": Integer(500, 700),

        f"{classifier_name}__dense_units": Categorical([
            "64"
        ]),

        f"{classifier_name}__dropout": Categorical([0.1]),
        f"{classifier_name}__spatial_dropout": Categorical([0.1]),
        f"{classifier_name}__learning_rate": Real(1e-6, 1e-4, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([32]),
        f"{classifier_name}__epochs": Categorical([100]),
        f"{classifier_name}__patience": Categorical([15]),

        # Multi-task learning
        f"{classifier_name}__uncertainty_weighting": Categorical([False]),
        f"{classifier_name}__fixed_gesture_weight": Categorical([1.0]),
        f"{classifier_name}__fixed_orientation_weight": Categorical([0.3]),
        f"{classifier_name}__fixed_phase_weight": Categorical([1.0]),

        # Augmentation
        f"{classifier_name}__use_mixup": Categorical([False]),
    }
    param_space = utils.prepare_multitask_param_space(
            param_space,
            search_mode,
            Categorical=Categorical,
    )

elif search_mode == "grid":
    
    param_grid = {
    # ============================================================
    # 1. Sequence builder (fixed to known good values)
    # ============================================================
    f"{pipe_name}__acc_modes": [("smoothed", "velocity", "displacement", "jerk")],
    f"{pipe_name}__rotation_modes": [("quaternion", "euler", "rot6d", "angular_velocity")],
    f"{pipe_name}__sampling_rate": [10],               # successful single‑task value
    f"{pipe_name}__interp_mode": ["ffill"],
    f"{pipe_name}__standardize": ["mean_std"],
    f"{pipe_name}__linear_acc_mode": ["baseline"],
    f"{pipe_name}__use_acc_magnitude": [True],
    f"{pipe_name}__use_linear_acc_magnitude": [True],
    f"{pipe_name}__tof_mode": ["pooled_stats"],
    f"{pipe_name}__tof_fill_mode": ["far_255"],
    f"{pipe_name}__thm_mode": ["centered_diff"],
    f"{pipe_name}__motion_filter_mode": ["extended_kalman"],
    # fixed Kalman params from your best Bayesian run
    f"{pipe_name}__kalman_process_noise": [0.031839],
    f"{pipe_name}__kalman_measurement_noise": [0.461918],
    f"{pipe_name}__use_dead_reckoning": [True],
    f"{pipe_name}__clip_value": [50.0],
    f"{pipe_name}__window_size": [66],
    f"{pipe_name}__smooth_alpha": [0.8],

    # ============================================================
    # 2. Multi‑task classifier – architecture & training
    # ============================================================
    # ----- Model size (3 levels, based on single‑task success) -----
    f"{classifier_name}__branch_filters": [
        {"acc": "64-128-128", "rot": "32-64", "tof": "32", "thm": "8"},   # Small
        {"acc": "128-256-256", "rot": "64-128", "tof": "64", "thm": "16"}, # Medium (best single‑task)
        {"acc": "256-512-512", "rot": "128-256", "tof": "128", "thm": "32"}, # Large
    ],
    # ----- Kernel sizes (same pattern as single‑task) -----
    f"{classifier_name}__branch_kernel_sizes": [
        {"acc": "5-3", "rot": "5-3", "tof": "5", "thm": "3"}
    ],
    # ----- No pooling – keep time dims equal -----
    f"{classifier_name}__branch_pool_sizes": [
        {"acc": "none", "rot": "none", "tof": "none", "thm": "none"}
    ],

    # ----- Fusion -----
    f"{classifier_name}__fusion_mode": ["attention"],
    f"{classifier_name}__gru_units": [160],      # only used if fusion_mode='bigru'
    f"{classifier_name}__attention_heads": [12],    # only used if fusion_mode='attention'

    # ----- Regularisation -----
    f"{classifier_name}__dropout": [0.01],
    f"{classifier_name}__spatial_dropout": [0.01],
    f"{classifier_name}__use_batch_norm": [True],

    # ----- Training hyperparameters -----
    f"{classifier_name}__learning_rate": [1e-4],
    f"{classifier_name}__batch_size": [32],
    f"{classifier_name}__epochs": [100],
    f"{classifier_name}__patience": [20],
    f"{classifier_name}__dense_units": ["64"],        # one dense layer before heads

    # ----- Multi‑task weighting (keep orientation weight low) -----
    f"{classifier_name}__uncertainty_weighting": [False],
    f"{classifier_name}__fixed_gesture_weight": [1.0],
    f"{classifier_name}__fixed_orientation_weight": [0.3],
    f"{classifier_name}__fixed_phase_weight": [1.0],

    # ----- Augmentation (turn off for now – add later if needed) -----
    f"{classifier_name}__use_mixup": [False],
    f"{classifier_name}__use_gaussian_noise": [False],
    f"{classifier_name}__use_magnitude_scaling": [False],
    f"{classifier_name}__use_time_mask": [False],
    f"{classifier_name}__use_time_shift": [False],
    f"{classifier_name}__use_channel_dropout": [False],
    f"{classifier_name}__use_modality_dropout": [False],
}

In [9]:
# ============================================================
# CV search
# ============================================================

cv = GroupKFold(n_splits=n_splits)

groups = target_only_train_df["subject"]
X_train = target_only_train_df.copy()
y_train = target_only_train_df[["sequence_id", primary_target, orientation_target, phase_target]].copy()

if search_mode == "bayesian":
    if BayesSearchCV is None:
        raise ImportError("skopt is not installed. Use search_mode='grid' or install scikit-optimize.")

    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        cv=cv,
        scoring=None,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        cv=cv,
        scoring=None,
        n_jobs=n_jobs,
        verbose=4,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

search.fit(X_train, y_train, groups=groups)

print("best gesture macro F1:", search.best_score_)
print("best params:")
print(search.best_params_)


Fitting 2 folds for each of 1 candidates, totalling 2 fits


I0000 00:00:1779903435.761219      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779903435.767289      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1779903446.675939      70 service.cc:152] XLA service 0x7a324c003150 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779903446.675973      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779903446.675976      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779903448.595064      70 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-27 17:37:33.812099: E external/local_xla/xla/service/slow_operation

[CV 1/2] END classifier__attention_heads=47, classifier__batch_size=32, classifier__branch_filters={"acc": "256-512-512-512", "rot": "256-512-768-768", "thm": "32", "tof": "256-256"}, classifier__branch_kernel_sizes={"acc": "5-5-5-5", "rot": "5-5-5-5", "thm": "3", "tof": "3-3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=64, classifier__dropout=0.1, classifier__epochs=100, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=0.3, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=536, classifier__learning_rate=1.8373473033929418e-05, classifier__maxlen=33, classifier__patience=15, classifier__spatial_dropout=0.1, classifier__uncertainty_weighting=False, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, se

2026-05-27 18:54:53.386274: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-27 18:54:53.727051: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


[CV 1/2] END classifier__attention_heads=82, classifier__batch_size=32, classifier__branch_filters={"acc": "256-512-512-512", "rot": "256-512-768-768", "thm": "32", "tof": "256-256"}, classifier__branch_kernel_sizes={"acc": "5-5-5-5", "rot": "5-5-5-5", "thm": "3", "tof": "3-3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=64, classifier__dropout=0.1, classifier__epochs=100, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=0.3, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=551, classifier__learning_rate=3.3859441208872077e-06, classifier__maxlen=88, classifier__patience=15, classifier__spatial_dropout=0.1, classifier__uncertainty_weighting=False, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, se

2026-05-27 19:06:40.321966: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-27 19:06:40.648887: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


[CV 2/2] END classifier__attention_heads=82, classifier__batch_size=32, classifier__branch_filters={"acc": "256-512-512-512", "rot": "256-512-768-768", "thm": "32", "tof": "256-256"}, classifier__branch_kernel_sizes={"acc": "5-5-5-5", "rot": "5-5-5-5", "thm": "3", "tof": "3-3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=64, classifier__dropout=0.1, classifier__epochs=100, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=0.3, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=551, classifier__learning_rate=3.3859441208872077e-06, classifier__maxlen=88, classifier__patience=15, classifier__spatial_dropout=0.1, classifier__uncertainty_weighting=False, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, se

2026-05-27 20:22:13.148084: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-27 20:22:13.502702: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


[CV 1/2] END classifier__attention_heads=59, classifier__batch_size=32, classifier__branch_filters={"acc": "256-512-512-512", "rot": "256-512-768-768", "thm": "32", "tof": "256-256"}, classifier__branch_kernel_sizes={"acc": "5-5-5-5", "rot": "5-5-5-5", "thm": "3", "tof": "3-3"}, classifier__branch_pool_sizes={"acc": "none", "rot": "none", "thm": "none", "tof": "none"}, classifier__dense_units=64, classifier__dropout=0.1, classifier__epochs=100, classifier__fixed_gesture_weight=1.0, classifier__fixed_orientation_weight=0.3, classifier__fixed_phase_weight=1.0, classifier__fusion_mode=attention, classifier__gru_units=638, classifier__learning_rate=1.4195928013966814e-06, classifier__maxlen=90, classifier__patience=15, classifier__spatial_dropout=0.1, classifier__uncertainty_weighting=False, classifier__use_batch_norm=True, classifier__use_mixup=False, sequence_builder__acc_modes=smoothed|velocity|displacement|jerk, sequence_builder__clip_value=50.0, sequence_builder__interp_mode=ffill, se

In [10]:
# ============================================================
# Save CV results
# ============================================================

cv_results_df = pd.DataFrame(search.cv_results_)
cv_path = results_dir / f"{search_mode}_multitask_hardsharing_cv_results_{timestamp}.csv"
cv_results_df.to_csv(cv_path, index=False)

best_params_path = results_dir / f"{search_mode}_multitask_hardsharing_best_params_{timestamp}.json"
with open(best_params_path, "w") as f:
    json.dump(search.best_params_, f, indent=2, default=str)

print(cv_path)
print(best_params_path)


results/bayesian_multitask_hardsharing_cv_results_20260527_1734.csv
results/bayesian_multitask_hardsharing_best_params_20260527_1734.json


In [11]:
# ============================================================
# Holdout evaluation with valid orientation/action/position decoding
# ============================================================

best_model = search.best_estimator_

if target_only_holdout_df is not None and not target_only_holdout_df.empty:
    X_holdout = target_only_holdout_df.copy()

    holdout_seq = (
        X_holdout
        .drop_duplicates("sequence_id")
        [["sequence_id", primary_target, orientation_target, position_target, "gesture"]]
        .reset_index(drop=True)
    )

    X_holdout_features = best_model[:-1].transform(X_holdout)
    clf = best_model.named_steps[classifier_name]

    pred_df = clf.predict_all(X_holdout_features)

    pred_df = pred_df.rename(
        columns={
            "phase_pred": "gesture_position_pred"
        }
    )

    proba = clf.predict_proba(X_holdout_features)

    orientation_proba_df = pd.DataFrame(
        proba["orientation"],
        columns=clf.orientation_classes_,
    )

    action_proba_df = pd.DataFrame(
        proba["gesture"],
        columns=clf.gesture_classes_,
    )

    position_proba_df = pd.DataFrame(
        proba["phase"],
        columns=clf.phase_classes_,
    )

    valid_table = (
        train_df.loc[
            train_df["sequence_type"].eq("Target"),
            ["orientation", "gesture", "gesture_action", "gesture_position"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    eps = 1e-9
    decoded_gestures = []

    for i in range(len(pred_df)):
        scores = []

        for _, row in valid_table.iterrows():
            ori = row["orientation"]
            act = row["gesture_action"]
            pos = row["gesture_position"]

            score = (
                np.log(orientation_proba_df.loc[i, ori] + eps)
                + np.log(action_proba_df.loc[i, act] + eps)
                + np.log(position_proba_df.loc[i, pos] + eps)
            )

            scores.append(score)

        decoded_gestures.append(
            valid_table.iloc[int(np.argmax(scores))]["gesture"]
        )

    holdout_pred = pd.concat(
        [
            holdout_seq.reset_index(drop=True),
            pred_df.reset_index(drop=True),
        ],
        axis=1,
    )

    holdout_pred["raw_reconstructed_gesture_pred"] = (
        holdout_pred["gesture_position_pred"].astype(str)
        + " - "
        + holdout_pred["gesture_action_pred"].astype(str)
    )

    holdout_pred["valid_decoded_gesture_pred"] = decoded_gestures

    gesture_action_f1 = f1_score(
        holdout_pred[primary_target],
        holdout_pred["gesture_action_pred"],
        average="macro",
    )

    gesture_position_f1 = f1_score(
        holdout_pred[position_target],
        holdout_pred["gesture_position_pred"],
        average="macro",
    )

    orientation_f1 = f1_score(
        holdout_pred[orientation_target],
        holdout_pred["orientation_pred"],
        average="macro",
    )

    raw_reconstructed_gesture_f1 = f1_score(
        holdout_pred["gesture"],
        holdout_pred["raw_reconstructed_gesture_pred"],
        average="macro",
    )

    valid_decoded_gesture_f1 = f1_score(
        holdout_pred["gesture"],
        holdout_pred["valid_decoded_gesture_pred"],
        average="macro",
    )

    print("holdout gesture_action macro F1:", round(gesture_action_f1, 4))
    print("holdout gesture_position macro F1:", round(gesture_position_f1, 4))
    print("holdout orientation macro F1:", round(orientation_f1, 4))
    print("holdout raw reconstructed gesture macro F1:", round(raw_reconstructed_gesture_f1, 4))
    print("holdout valid decoded gesture macro F1:", round(valid_decoded_gesture_f1, 4))

    print("\nGesture action report")
    print(classification_report(
        holdout_pred[primary_target],
        holdout_pred["gesture_action_pred"],
    ))

    print("\nGesture position report")
    print(classification_report(
        holdout_pred[position_target],
        holdout_pred["gesture_position_pred"],
    ))

    print("\nOrientation report")
    print(classification_report(
        holdout_pred[orientation_target],
        holdout_pred["orientation_pred"],
    ))

    print("\nRaw reconstructed full gesture report")
    print(classification_report(
        holdout_pred["gesture"],
        holdout_pred["raw_reconstructed_gesture_pred"],
    ))

    print("\nValid decoded full gesture report")
    print(classification_report(
        holdout_pred["gesture"],
        holdout_pred["valid_decoded_gesture_pred"],
    ))

    holdout_path = results_dir / f"multitask_position_holdout_predictions_{timestamp}.csv"
    summary_path = results_dir / f"multitask_position_holdout_summary_{timestamp}.csv"

    holdout_pred.to_csv(holdout_path, index=False)

    pd.DataFrame([
        {
            "search_mode": search_mode,
            "best_cv_gesture_action_macro_f1": search.best_score_,
            "holdout_gesture_action_macro_f1": gesture_action_f1,
            "holdout_gesture_position_macro_f1": gesture_position_f1,
            "holdout_orientation_macro_f1": orientation_f1,
            "holdout_raw_reconstructed_gesture_macro_f1": raw_reconstructed_gesture_f1,
            "holdout_valid_decoded_gesture_macro_f1": valid_decoded_gesture_f1,
            "n_train_sequences": target_only_train_df["sequence_id"].nunique(),
            "n_holdout_sequences": target_only_holdout_df["sequence_id"].nunique(),
        }
    ]).to_csv(summary_path, index=False)

    print(holdout_path)
    print(summary_path)

else:
    print("No target-only holdout dataframe available. Skipping holdout evaluation.")

holdout gesture_action macro F1: 0.5674
holdout gesture_position macro F1: 0.5524
holdout orientation macro F1: 0.9315
holdout raw reconstructed gesture macro F1: 0.1912
holdout valid decoded gesture macro F1: 0.4803

Gesture action report
               precision    recall  f1-score   support

   pinch skin       0.61      0.40      0.48       162
    pull hair       0.66      0.74      0.70       243
pull hairline       0.52      0.56      0.54        81
      scratch       0.53      0.58      0.55       162

     accuracy                           0.59       648
    macro avg       0.58      0.57      0.57       648
 weighted avg       0.59      0.59      0.59       648


Gesture position report
              precision    recall  f1-score   support

   Above ear       0.89      0.51      0.65        81
       Cheek       0.47      0.43      0.45        81
     Eyebrow       0.34      0.27      0.30        81
     Eyelash       0.43      0.37      0.40        81
    Forehead       0.

In [12]:
# ============================================================
# Inspect task setup
# ============================================================

clf = search.best_estimator_.named_steps[classifier_name]

print("gesture target:", clf.gesture_target)
print("orientation target:", clf.orientation_target)
print("third head target:", clf.phase_target)

print("uncertainty weighting:", clf.uncertainty_weighting)
print("fixed gesture weight:", clf.fixed_gesture_weight)
print("fixed orientation weight:", clf.fixed_orientation_weight)
print("fixed position/phase weight:", clf.fixed_phase_weight)

if hasattr(clf, "gesture_classes_"):
    print("\ngesture_action classes:")
    print(clf.gesture_classes_)

if hasattr(clf, "orientation_classes_"):
    print("\norientation classes:")
    print(clf.orientation_classes_)

if hasattr(clf, "phase_classes_"):
    print("\ngesture_position classes:")
    print(clf.phase_classes_)

gesture target: gesture_action
orientation target: orientation
third head target: gesture_position
uncertainty weighting: False
fixed gesture weight: 1.0
fixed orientation weight: 0.3
fixed position/phase weight: 1.0

gesture_action classes:
['pinch skin' 'pull hair' 'pull hairline' 'scratch']

orientation classes:
['Lie on Back' 'Lie on Side - Non Dominant'
 'Seated Lean Non Dom - FACE DOWN' 'Seated Straight']

gesture_position classes:
['Above ear' 'Cheek' 'Eyebrow' 'Eyelash' 'Forehead' 'Neck']
